Gold Layer data preparation for Analytics

In [0]:
# 1. Setup and load Silver data

storage_account_name = "REMOVED"
storage_account_key = "REMOVED"

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "True")

# Define paths

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"


# Read silver data (Delat Format)

print("Loading silver data")

df_silver_trans = spark.read.format("delta").load(f"{silver_path}/transactions")
df_silver_units = spark.read.format("delta").load(f"{silver_path}/units")

print("Silver data Loaded and Gold configured")


In [0]:
# 2. Gold Table 1. Monthly Sales Trend
from pyspark.sql.functions import year, month, sum, count, avg, round

# Aggregate as total amount and count of sales per month

gold_sales_overview = df_silver_trans.groupBy(
    year("transaction_date").alias("year"),
    month("transaction_date").alias("month")
    ).agg(
        sum("price").alias("total_sales_aed"),
        count("transaction_id").alias("total_transactions"),
        avg("price_per_sqft").alias("avg_price_sqft")
    ).orderBy("year","month")

# Rounding numbers for cleaner reports

gold_sales_overview = gold_sales_overview.withColumn("total_sales_aed",round("total_sales_aed",2))
gold_sales_overview = gold_sales_overview.withColumn("avg_price_sqft",round("avg_price_sqft",2))

print("Sales overview created")

display(gold_sales_overview.limit(10))


In [0]:
# 3. Gold Table 2 - Area performance
from pyspark.sql.functions import sum, count, avg, round, col, desc
# Highest sales per area

gold_area_stats = df_silver_trans.groupBy("area_name").agg(
    sum("price").alias("total_price_aed"),
    count("transaction_id").alias("total_transactions"),
    avg("price_per_sqft").alias("avg_sqft_price")
)

# Clean the data

gold_area_stats = gold_area_stats.withColumn("total_price_aed",round("total_price_aed",0))
gold_area_stats = gold_area_stats.withColumn("avg_sqft_price",round("avg_sqft_price",2))

print("Area stats created")

display(gold_area_stats.orderBy(desc("avg_sqft_price")).limit(10))


In [0]:
# 4. Gold Tbale 3 - Supply and Inventory
from pyspark.sql.functions import count, avg, sum, desc, round
# Units by Bedroom type

gold_supply_stats = df_silver_units.groupBy("project_number","room_type","bedrooms").agg(
    count("unit_number").alias("unit_count"),
    round(avg("unit_area"), 2).alias("avg_unit_area")
).orderBy(desc("unit_count"))

print("Supply stats created")

display(gold_supply_stats.limit(10))

In [0]:
# 5. Write Gold tables to gold layer ADLS

print("Writting gold tables")

# Writting sales_overview table
gold_sales_overview.write.format("delta").mode("overwrite").save(f"{gold_path}/sales_overview")

# writting area_stats table
gold_area_stats.write.format("delta").mode("overwrite").save(f"{gold_path}/area_stats")

# writting supply_stats table
gold_supply_stats.write.format("delta").mode("overwrite").save(f"{gold_path}/supply_stats")

print("GOLD tables are written to gold layer.")

In [0]:
# 6. Verify Gold data
try:
    df_sales_validation = spark.read.format("delta").load(f"{gold_path}/sales_overview")
    count = df_sales_validation.count()

    if count > 0 :
        print("Sales Overview has data")
    else :
        print("No data found in Sales Overview gold layer table")
except Exception as e :
    print(f"Failed reading data from gold layer table Sales Overview with Error : {e}")
